# 🕵️‍♀️ Relaciones Ocultas: Lógica de Predicados para Detectar Nepotismo (con `kanren`)

## 🎯 Objetivo de aprendizaje
Modelar **relaciones familiares** como una **base de hechos** y usar **reglas lógicas** para **inferir** relaciones ocultas (hermanos, tíos, primos) y luego cruzarlas con una **nómina** (jefe–empleado) para detectar posibles casos de nepotismo.

---

## 🧠 Idea clave: Proposicional vs Predicados

### Lógica proposicional (más rígida)
- “Juan es corrupto” → es una proposición (V/F) pero **no explica** por qué, ni permite explorar casos similares.

### Lógica de predicados (más expresiva)
- “Existe un X tal que X robó dinero” → permite razonar con **variables** y **relaciones**:
  - `robó(X, dinero)`
  - `padre(P, X)`
  - `jefe(J, X)`

📌 En este notebook NO vamos a afirmar “alguien es corrupto”.
Vamos a detectar **relaciones familiares** dentro de una estructura laboral usando reglas claras.


## 🧰 Herramienta: `kanren` (programación lógica)
`kanren` permite hacer **inferencia** a partir de:
- **Hechos** (facts)
- **Relaciones** (relations)
- **Reglas** (rules)

Instalación:
```bash
pip install kanren
```

> Si tu entorno no permite instalar, alternativa: lógica de grafos (NetworkX).  
Aquí usaremos `kanren` para ver inferencia real.


In [9]:
!pip install kanren
!python.exe -m pip install --upgrade pip
!pip install --upgrade --force-reinstall kanren

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
  Using cached kanren-0.3.0-py3-none-any.whl.metadata (6.2 kB)
  Using cached toolz-1.1.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached multipledispatch-1.0.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached unification-0.3.1-py3-none-any.whl.metadata (2.2 kB)
Using cached kanren-0.3.0-py3-none-any.whl (17 kB)
Using cached unification-0.3.1-py3-none-any.whl (7.0 kB)
Using cached multipledispatch-1.0.0-py3-none-any.whl (12 kB)
Using cached toolz-1.1.0-py3-none-any.whl (58 kB)

  Attempting uninstall: unification

    Found existing installation: unification 0.3.1

    Uninstalling unification-0.3.1:

      Successfully uninstalled unification-0.3.1

  Attempting uninstall: multipledispatch

    Found existing installation: multipledispatch 1.0.0

    U

## 1) INICIO — Dinámica: “El Abogado del Diablo”
Ejemplo de ambigüedad:

- “Juan es corrupto” (no es inferible, es una etiqueta)
- “Existe un X tal que X desvió recursos” (es verificable si hay evidencias)

🔎 En IA responsable, **lo que podemos inferir** debe basarse en datos y reglas, no en etiquetas.


## 2) DESARROLLO — El reto: Detectando nepotismo

### Base de hechos
Tendremos:
- Una lista de empleados
- La relación **padre/madre → hijo**
- Una nómina con pares **(jefe, empleado)**

### Misión
Escribir reglas lógicas para inferir:
- **Hermanos**
- **Tíos**
- **Primos**

Y luego cruzarlo con la nómina para detectar nepotismo.


## ✅ Paso A — Preparar datos (hechos)


In [10]:
# Hechos (base de datos) para prueba en clase.
# Puedes cambiar nombres por casos de tu contexto (sin datos sensibles reales).

# Empleados/Personas
personas = [
    "ana", "beto", "carla", "diego", "elena",
    "fabi", "gabo", "hector", "irene", "juan"
]

# Relación (padre_o_madre, hijo)
# Ejemplo: ana y beto son padres de carla; ana también es madre de diego, etc.
padres = [
    ("ana", "carla"),
    ("beto", "carla"),
    ("ana", "diego"),
    ("beto", "diego"),
    ("elena", "fabi"),
    ("hector", "fabi"),
    ("elena", "gabo"),
    ("hector", "gabo"),
    ("irene", "juan"),
]

# Nómina: (jefe, empleado)
nomina = [
    ("carla", "fabi"),
    ("diego", "gabo"),
    ("ana", "juan"),
    ("elena", "carla"),
    ("hector", "diego"),
]


## ✅ Paso B — Crear relaciones en `kanren` y cargar hechos


In [11]:
from kanren import Relation, facts, run, var, conde, eq

# Definimos la relación padre(P, H) = P es padre/madre de H
padre = Relation()
facts(padre, *padres)

# Definimos la relación jefe(J, E) = J es jefe de E
jefe = Relation()
facts(jefe, *nomina)

print("Hechos cargados: padre(P,H) y jefe(J,E)")


Hechos cargados: padre(P,H) y jefe(J,E)


In [12]:
pip show kanren

Name: kanren
Version: 0.3.0
Summary: Logic Programming in python
Home-page: http://github.com/logpy/logpy
Author: Matthew Rocklin
Author-email: mrocklin@gmail.com
License: BSD
Location: C:\Users\OSCAR\AppData\Roaming\Python\Python312\site-packages
Requires: multipledispatch, toolz, unification
Required-by: 
Note: you may need to restart the kernel to use updated packages.


## ✅ Paso C — Reglas lógicas (inferencia)

### 1) Hermanos
Dos personas X e Y son hermanos si comparten al menos un padre/madre.

> hermano(X, Y) ⇔ ∃P: padre(P, X) ∧ padre(P, Y) ∧ X ≠ Y


In [29]:
from kanren import Relation, facts, run, var, lall, conde, eq
from kanren.core import fail, succeed
def neqo(a, b):
    return conde([eq(a, b), fail], [succeed])

def hermano(x, y):
    p = var()
    return lall(
        padre(p, x),
        padre(p, y),
        neqo(x, y)    
    )

ImportError: cannot import name 'succeed' from 'kanren.core' (C:\Users\OSCAR\AppData\Roaming\Python\Python312\site-packages\kanren\core.py)

### 2) Tío / Tía
u es tío de y si u es hermano de algún padre/madre de y.

> tio(U, Y) ⇔ ∃P: padre(P, Y) ∧ hermano(U, P)


In [24]:
def tio(t, s):
    p = var()
    return lall(
        padre(p, s),
        hermano(t, p)
    )

### 3) Primo / Prima
x es primo de y si los padres/madres de x y y son hermanos.

> primo(X, Y) ⇔ ∃Px, Py: padre(Px, X) ∧ padre(Py, Y) ∧ hermano(Px, Py)


In [25]:
def primo(a, b):
    pa, pb = var(), var()
    return lall(
        padre(pa, a),
        padre(pb, b),
        hermano(pa, pb),
        neqo(a, b)     # 👈 en lugar de neq(a, b)
    )

## 🔍 Probar inferencias (consultas)

En programación lógica, en vez de “calcular”, **preguntamos**:
- ¿Quiénes son hermanos de Diego?
- ¿Quién es tío de Fabi?
- ¿Quiénes son primos de Carla?


In [26]:
x = var()
print("Hermanos de diego:", run(10, x, hermano(x, "diego")))
print("Tíos de fabi:", run(10, x, tio(x, "fabi")))
print("Primos de carla:", run(10, x, primo(x, "carla")))


NameError: name 'neqo' is not defined

## 🧾 Detectar nepotismo (cruce con nómina)

### Pseudocódigo conceptual
```python
def es_nepotismo(jefe, empleado):
    if es_familiar_directo(jefe, empleado):
        return True
    return False
```

### En esta actividad
Definimos “familiar” como:
- Padre/Madre (padre(jefe, empleado) o padre(empleado, jefe))
- Hermanos
- Tíos
- Primos

📌 Puedes discutir en clase qué relaciones deben contar como “conflicto de interés” según la norma/institución.


In [19]:
def es_familiar(j, e):
    # Consulta lógica: devuelve True si existe alguna prueba (run) de relación
    # (kanren opera con metas; aquí preguntamos si hay soluciones)
    return (
        run(1, var(), padre(j, e)) != () or
        run(1, var(), padre(e, j)) != () or
        run(1, var(), hermano(j, e)) != () or
        run(1, var(), tio(j, e)) != () or
        run(1, var(), tio(e, j)) != () or
        run(1, var(), primo(j, e)) != ()
    )

posibles = []
for j, e in nomina:
    posibles.append((j, e, es_familiar(j, e)))

posibles


[('carla', 'fabi', False),
 ('diego', 'gabo', False),
 ('ana', 'juan', False),
 ('elena', 'carla', False),
 ('hector', 'diego', False)]

### Presentar resultados como tabla (auditables)


In [ ]:
import pandas as pd

df = pd.DataFrame(posibles, columns=["jefe", "empleado", "posible_nepotismo"])
df


## 3) CIERRE — Visualización y reflexión

### Visualización (pizarrón)
Dibuja el **árbol genealógico** que el software “descubrió” usando:
- Hechos (padre/madre)
- Reglas (hermano, tío, primo)

### Reflexión crítica
- ✔️ Ventaja: reglas explícitas, trazables, auditables.
- ❌ Riesgo: si los datos están incompletos o mal capturados, la inferencia falla.
- ⚖️ Ética: detectar *posible* nepotismo no es “probar corrupción”; es **señal de alerta** para auditoría.

🔜 Siguiente paso:
- Integrar más hechos (cónyuges, apellidos, domicilios)
- Combinar con grafos (NetworkX) y métricas de centralidad
- Diseñar criterios institucionales (qué cuenta como conflicto de interés)
